En este cuaderno vamos a usar un modelo ViT (Vision Transformers). El elegido es: *google/vit-base-patch16-224-in21k*

In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset, Image, Features, Value
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate

# 1. Fase de Entrenamiento

## 1.1 Defiendo las rutas y el modelo con el que vamos a trabajar

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
CSV_IMAGENES = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes.csv"
CSV_TRAIN_MASTER_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv"
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"

OUTPUT_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/ViT_FineTuned"
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"

In [5]:
print("Cargando el dataset de imágenes y las plantillas de texto...")
df_imagenes = pd.read_csv(CSV_IMAGENES)
df_train_text = pd.read_csv(CSV_TRAIN_MASTER_TEXT)
df_test_text = pd.read_csv(CSV_TEST_TEXT)

Cargando el dataset de imágenes y las plantillas de texto...


## 1.2 Particionado

**OJO** con este paso, tendremos que tener en cuenta el contenido de la columna *id_EXIST* porque estaríamos falseando los resultados si de un mismo vídeo tenemos un frame en el conjunto de entrenamiento, otro frame en el conjunto de test y otro frame en el conjunto de validación.

In [6]:
# =================================================================
# FASE 1: Alineación 80/20 Estática Multimodal
# =================================================================
# Extraemos los IDs de los vídeos para que el particionado de imágenes
# sea EXACTAMENTE idéntico al de texto.
train_master_ids = df_train_text['id_EXIST'].unique()
test_ids = df_test_text['id_EXIST'].unique()

# Filtramos las imágenes basándonos en las particiones estáticas
df_train_master = df_imagenes[df_imagenes['id_EXIST'].isin(train_master_ids)].copy()
test_df = df_imagenes[df_imagenes['id_EXIST'].isin(test_ids)].copy()

# =================================================================
# FASE 2: División Dinámica del Train Master (90% Train / 10% Valid)
# =================================================================
# Usamos GroupShuffleSplit para asegurar que todos los fotogramas
# de un mismo vídeo caen en el MISMO bloque (evita Data Leakage)
gss_train_val = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)

# Separamos Train (90%) y Valid (10%)
train_idx, val_idx = next(gss_train_val.split(df_train_master, groups=df_train_master['id_EXIST']))

train_df = df_train_master.iloc[train_idx].copy()
val_df = df_train_master.iloc[val_idx].copy()

print("\n--- DISTRIBUCIÓN DEL DATASET DE IMÁGENES ---")
print(f"Vídeos en Train: {train_df['id_EXIST'].nunique()} (Aprox {len(train_df)} fotogramas)")
print(train_df['label'].value_counts())

print(f"\nVídeos en Valid: {val_df['id_EXIST'].nunique()} (Aprox {len(val_df)} fotogramas)")
print(val_df['label'].value_counts())

print(f"\nVídeos en Test (Intocable): {test_df['id_EXIST'].nunique()} (Aprox {len(test_df)} fotogramas)")
print(test_df['label'].value_counts())


--- DISTRIBUCIÓN DEL DATASET DE IMÁGENES ---
Vídeos en Train: 1805 (Aprox 7218 fotogramas)
label
0    3750
1    3468
Name: count, dtype: int64

Vídeos en Valid: 201 (Aprox 804 fotogramas)
label
0    428
1    376
Name: count, dtype: int64

Vídeos en Test (Intocable): 502 (Aprox 2008 fotogramas)
label
0    1044
1     964
Name: count, dtype: int64


## 1.3 Conversión a Formato `Dataset` de Hugging Face

In [7]:
def crear_dataset(dataframe):
    # Primero creamos el dataset leyendo las rutas como texto plano desde el dataframe
    dataset = Dataset.from_pandas(dataframe)

    # Casteamos la columna de texto a tipo Image() para que lea los archivos de Drive
    dataset = dataset.cast_column("path_imagen", Image())

    # Renombramos para que el modelo lo entienda
    return dataset.rename_column("path_imagen", "image")

print("Transformando rutas en imágenes procesables...")
train_dataset = crear_dataset(train_df)
valid_dataset = crear_dataset(val_df)

Transformando rutas en imágenes procesables...


## 1.4 Procesador de Imágenes (`ViTImageProcessor`)

Esto es igual que el tokenizador de texto, pero para imágenes (redimensiona a 224x224, normaliza colores, etc.)

In [8]:
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def process_images(batch):
    # Toma una lista de imágenes y las convierte en los tensores matemáticos que ViT necesita
    inputs = processor([img.convert("RGB") for img in batch["image"]], return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

# Aplicamos la transformación "al vuelo" para no saturar la RAM
train_dataset.set_transform(process_images)
valid_dataset.set_transform(process_images)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

## 1.5 Inicialización del Modelo Base ViT

In [9]:
id2label = {0: "No misógino", 1: "Misógino"}
label2id = {"No misógino": 0, "Misógino": 1}

model = ViTForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # Necesario porque el modelo in21k original no tiene capa de clasificación
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.weight   | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.weight    | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.bias             | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.key.bias     | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.bias      | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.bias            | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.

## 1.6 Métrica de Evaluación

In [10]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

# Usamos tu amado F1 Score macro
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

## 1.7 Hiperparámetros del Entrenamiento

In [11]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # IMPORTANTE: En visión debe ser False
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,          # ViT prefiere Learning Rates más bajos que los LLMs
    per_device_train_batch_size=16, # ViT es ligero, aguanta un batch de 16 en Colab
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,                   # Precisión mixta para acelerar en GPU
    report_to="none"
)

## 1.8 Entrenamiento

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Iniciando entrenamiento visual con ViT...")
trainer.train()

print("💾 Guardando modelo visual...")
trainer.save_model(OUTPUT_DIR + "/modelo_final")
processor.save_pretrained(OUTPUT_DIR + "/modelo_final")
print("✅ ¡Entrenamiento completado!")

🚀 Iniciando entrenamiento visual con ViT...


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# 2. Fase de Inferencia/Evaluación

## 2.1 Max-Pooling

Con que solamente 1 solo fotograma de los 4 sea etiquetado como misógino, el video entero se marcará como misógino.

In [ ]:
print("Hola mundo")

# 2.2 Average Pooling

Se cogerá el porcentaje de seguridad del modelo para los 4 fotogramas y haremos la media. Si la media >= 50% se cataloga vídeo como misógino.

In [ ]:
print("Hola mundo")

## 2.3 Votación por mayoría

Hacemos que al menos 2 o 3 fotogramas sean clasificados como misóginos para considerar el vídeo como misógino.

In [ ]:
print("Hola mundo")